# Notebook 03: Paralelismo, Benchmarks y Patrones Avanzados

**Módulo 16 — Clase 3**

---

In [ ]:
import asyncio
import time
import threading
import os
import sys
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

print(f'Python {sys.version}')
print(f'CPU cores disponibles: {os.cpu_count()}')

## Sección 1: create_task vs gather — trabajo intermedio gratis

In [ ]:
async def tarea_io(nombre: str, duracion: float) -> str:
    await asyncio.sleep(duracion)
    return f"{nombre} completada"

def trabajo_cpu_ligero(n: int) -> int:
    return sum(range(n))

print("=== gather: barrera ===")
t0 = time.perf_counter()
r1, r2 = await asyncio.gather(
    tarea_io("A", 1.0),
    tarea_io("B", 1.0),
)
t_gather = time.perf_counter() - t0
print(f"Resultados: {r1}, {r2}")
print(f"Tiempo: {t_gather:.2f}s (esperado ~1s)")
print()

print("=== create_task + trabajo intermedio ===")
t0 = time.perf_counter()

tarea_a = asyncio.create_task(tarea_io("A", 1.0))

t_trabajo = time.perf_counter()
resultado_cpu = trabajo_cpu_ligero(5_000_000)
t_trabajo = time.perf_counter() - t_trabajo
print(f"  Trabajo CPU completado en {t_trabajo:.2f}s (durante wait de A)")

resultado_a = await tarea_a
t_create_task = time.perf_counter() - t0

print(f"  {resultado_a}")
print(f"Tiempo total: {t_create_task:.2f}s (esperado ~max(1s, tiempo_cpu))")
print()
print(f"El trabajo CPU ({t_trabajo:.2f}s) fue 'gratis' — ocurrió durante el wait de A.")
print(f"Sin create_task, hubiera sumado {1.0 + t_trabajo:.2f}s en lugar de {t_create_task:.2f}s")

## Sección 2: as_completed — procesar resultados conforme llegan

In [ ]:
async def buscar_fuente(nombre: str, latencia: float) -> str:
    await asyncio.sleep(latencia)
    return f"resultado de {nombre} (tardó {latencia:.1f}s)"

fuentes = [
    ("Wikipedia", 0.5),
    ("ArXiv",     2.1),
    ("GitHub",    0.8),
    ("PubMed",    1.5),
    ("DuckDuck",  0.3),
]

print("=== asyncio.gather: espera al más lento ===")
t0 = time.perf_counter()
resultados = await asyncio.gather(
    *[buscar_fuente(nombre, lat) for nombre, lat in fuentes]
)
t_gather = time.perf_counter() - t0
print(f"Todos los resultados disponibles a los {t_gather:.2f}s")
for r in resultados:
    print(f"  {r}")
print()

print("=== asyncio.as_completed: procesa conforme llegan ===")
t0 = time.perf_counter()
tareas = [asyncio.create_task(buscar_fuente(n, l)) for n, l in fuentes]

async for tarea_completada in asyncio.as_completed(tareas):
    resultado = await tarea_completada
    t_llegada = time.perf_counter() - t0
    print(f"  t={t_llegada:.2f}s → {resultado}")

print()
print("El orden de llegada es por latencia, no por orden de creación.")
print("DuckDuck (0.3s) llega primero aunque fue creado al final.")

## Sección 3: asyncio.Queue — productor-consumidor

In [ ]:
N_PETICIONES = 10
RITMO_PRODUCCION = 0.5
T_PROCESAMIENTO = 0.5

async def productor(queue: asyncio.Queue, n: int, n_workers: int):
    for i in range(n):
        await asyncio.sleep(RITMO_PRODUCCION)
        await queue.put(i)
    for _ in range(n_workers):
        await queue.put(None)

async def worker(queue: asyncio.Queue, nombre: str, resultados: list):
    while True:
        item = await queue.get()
        if item is None:
            break
        await asyncio.sleep(T_PROCESAMIENTO)
        resultados.append(item)
        queue.task_done()

for n_workers in [1, 2, 3]:
    queue = asyncio.Queue()
    resultados = []
    t0 = time.perf_counter()
    await asyncio.gather(
        productor(queue, N_PETICIONES, n_workers),
        *[worker(queue, f'w{i}', resultados) for i in range(n_workers)]
    )
    t_total = time.perf_counter() - t0
    print(f'{n_workers} worker(s): {t_total:.2f}s para {N_PETICIONES} peticiones')

print()
print('Con más workers se reduce el cuello de botella en el procesamiento.')

## Sección 4: fire-and-forget — excepción silenciada vs tracked

In [ ]:
async def tarea_que_falla():
    await asyncio.sleep(0.1)
    raise ValueError("¡Error en la tarea!")

print("=== Anti-patrón: fire-and-forget ===")
t = asyncio.create_task(tarea_que_falla())
await asyncio.sleep(0.5)
print("No se vio ningún error — ¡la excepción fue silenciada!")
print()

print("=== Patrón correcto: tracking con add_done_callback ===")

errores_capturados = []

def manejar_resultado(fut):
    if fut.cancelled():
        print("  [callback] Tarea cancelada")
    elif fut.exception() is not None:
        exc = fut.exception()
        errores_capturados.append(exc)
        print(f"  [callback] Excepción capturada: {type(exc).__name__}: {exc}")
    else:
        print(f"  [callback] Tarea completada con resultado: {fut.result()}")

t2 = asyncio.create_task(tarea_que_falla())
t2.add_done_callback(manejar_resultado)
await asyncio.sleep(0.5)
print(f"Errores registrados: {len(errores_capturados)}")
print("El programa no se cae, pero tenemos visibilidad del error.")

## Sección 5: asyncio vs ThreadPoolExecutor — I/O-bound

In [ ]:
def io_bound_sync(duracion: float) -> str:
    time.sleep(duracion)
    return "resultado"

N = 20
DUR = 0.2

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=N) as pool:
    futuros = [pool.submit(io_bound_sync, DUR) for _ in range(N)]
    resultados = [f.result() for f in futuros]
t_thread = time.perf_counter() - t0
print(f"ThreadPoolExecutor ({N} workers): {t_thread:.2f}s  (esperado ~{DUR:.1f}s)")

async def io_bound_async(duracion: float) -> str:
    await asyncio.sleep(duracion)
    return "resultado"

t0 = time.perf_counter()
resultados_async = await asyncio.gather(*[io_bound_async(DUR) for _ in range(N)])
t_async = time.perf_counter() - t0
print(f"asyncio.gather       ({N} coroutines): {t_async:.2f}s  (esperado ~{DUR:.1f}s)")
print()
print(f"Diferencia: {abs(t_thread - t_async)*1000:.0f}ms")
print("asyncio es preferible para código nuevo (menor overhead, más claro).")
print("ThreadPoolExecutor es útil cuando la librería ya usa funciones síncronas (requests, psycopg2).")

## Sección 6: threading vs multiprocessing — CPU-bound con el GIL en números

In [ ]:
def tarea_cpu(n: int) -> int:
    return sum(range(n))

N_TRABAJO = 20_000_000
N_WORKERS_LIST = [1, 2, 4]

n = max(N_WORKERS_LIST)
t0 = time.perf_counter()
for _ in range(n): tarea_cpu(N_TRABAJO)
t_seq = time.perf_counter() - t0
print(f"Secuencial ({n} tareas): {t_seq:.2f}s (baseline)")
print()

print(f"{'Workers':<10} {'Threading':>12} {'Speedup':>10} {'ProcessPool':>14} {'Speedup':>10}")
print("-" * 60)

for nw in N_WORKERS_LIST:
    t0 = time.perf_counter()
    hilos = [threading.Thread(target=tarea_cpu, args=(N_TRABAJO,)) for _ in range(nw)]
    for h in hilos: h.start()
    for h in hilos: h.join()
    t_th = time.perf_counter() - t0

    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=nw) as pool:
        list(pool.map(tarea_cpu, [N_TRABAJO] * nw))
    t_pp = time.perf_counter() - t0

    sp_th = t_seq / t_th
    sp_pp = t_seq / t_pp
    print(f"{nw:<10} {t_th:>10.2f}s {sp_th:>9.2f}x {t_pp:>12.2f}s {sp_pp:>9.2f}x")

print()
print("Threading CPU-bound: speedup ~1x (GIL serializa todo)")
print("ProcessPool CPU-bound: speedup ~Nx (cada proceso tiene su propio GIL)")

## Sección 7: joblib vs ProcessPoolExecutor

In [ ]:
try:
    from joblib import Parallel, delayed
    JOBLIB_OK = True
except ImportError:
    print("joblib no instalado — pip install joblib")
    JOBLIB_OK = False

if JOBLIB_OK:
    datos = list(range(1_000_000, 1_001_000))

    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=4) as pool:
        resultados_ppe = list(pool.map(tarea_cpu, datos))
    t_ppe = time.perf_counter() - t0
    print(f"ProcessPoolExecutor (4 workers): {t_ppe:.2f}s")

    t0 = time.perf_counter()
    resultados_jl = Parallel(n_jobs=4)(delayed(tarea_cpu)(n) for n in datos)
    t_jl = time.perf_counter() - t0
    print(f"joblib.Parallel (n_jobs=4):       {t_jl:.2f}s")

    assert resultados_ppe == resultados_jl, "resultados diferentes!"
    print(f"\nResultados idénticos: ✓")
    print("\njoblib es preferible cuando: interfaz más simple, integración con numpy/sklearn,")
    print("backend 'loky' más robusto, o necesitas cambiar entre backends (threads/processes).")

## Sección 8: Anti-patrón lambda (PicklingError)

In [ ]:
try:
    with ProcessPoolExecutor(max_workers=2) as pool:
        resultados = list(pool.map(lambda x: x**2, [1, 2, 3, 4]))
    print("Sin error (puede ocurrir en algunos sistemas)")
    print(resultados)
except Exception as e:
    print(f"Error esperado: {type(e).__name__}: {e}")

print()

def al_cuadrado(x: int) -> int:
    return x ** 2

with ProcessPoolExecutor(max_workers=2) as pool:
    resultados_fn = list(pool.map(al_cuadrado, [1, 2, 3, 4]))
print(f"Fix con función a nivel de módulo: {resultados_fn}  ✓")

from functools import partial

def potencia(x: int, exp: int) -> int:
    return x ** exp

al_cubo = partial(potencia, exp=3)
with ProcessPoolExecutor(max_workers=2) as pool:
    resultados_partial = list(pool.map(al_cubo, [1, 2, 3, 4]))
print(f"Fix con functools.partial:         {resultados_partial}  ✓")
print()
print("Regla: ProcessPoolExecutor solo acepta funciones picklables (nivel de módulo o partial).")
print("Las lambdas y funciones anidadas NO son picklables.")

## Sección 9: Pool por petición vs pool compartido

In [ ]:
N_PETICIONES = 20

t0 = time.perf_counter()
for _ in range(N_PETICIONES):
    with ProcessPoolExecutor(max_workers=2) as pool:
        list(pool.map(tarea_cpu, [100_000, 100_000]))
t_pool_por_peticion = time.perf_counter() - t0
print(f"Pool por petición ({N_PETICIONES} veces): {t_pool_por_peticion:.2f}s")

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=2) as pool_compartido:
    for _ in range(N_PETICIONES):
        list(pool_compartido.map(tarea_cpu, [100_000, 100_000]))
t_pool_compartido = time.perf_counter() - t0
print(f"Pool compartido ({N_PETICIONES} peticiones): {t_pool_compartido:.2f}s")

print(f"\nOverhead del anti-patrón: {t_pool_por_peticion/t_pool_compartido:.1f}× más lento")
print("El overhead es el costo de crear/destruir procesos N veces en lugar de una.")

## Sección 10: run_in_executor — asyncio + ProcessPoolExecutor (M5b)

In [ ]:
N_USUARIOS = 10
T_IO = 0.1
T_CPU = 0.5

def inferencia_local(historial):
    time.sleep(T_CPU)
    return f"respuesta para {historial}"

async def peticion_v_a(user_id: int) -> dict:
    t0 = time.perf_counter()
    await asyncio.sleep(T_IO)
    time.sleep(T_CPU)
    return {'user': user_id, 'latencia': time.perf_counter() - t0}

async def peticion_v_b(user_id: int, executor) -> dict:
    t0 = time.perf_counter()
    await asyncio.sleep(T_IO)
    loop = asyncio.get_event_loop()
    await loop.run_in_executor(executor, inferencia_local, f'user_{user_id}')
    return {'user': user_id, 'latencia': time.perf_counter() - t0}

print(f"Escenario: {N_USUARIOS} usuarios, I/O={T_IO}s, CPU inferencia={T_CPU}s")
print()

t0 = time.perf_counter()
res_a = await asyncio.gather(*[peticion_v_a(i) for i in range(N_USUARIOS)])
t_a = time.perf_counter() - t0
print(f"Versión a (asyncio puro, time.sleep bloqueante): {t_a:.2f}s")
print(f"  Latencia promedio: {sum(r['latencia'] for r in res_a)/N_USUARIOS:.2f}s/usuario")

with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    t0 = time.perf_counter()
    res_b = await asyncio.gather(*[peticion_v_b(i, executor) for i in range(N_USUARIOS)])
    t_b = time.perf_counter() - t0

print(f"Versión b (asyncio + ProcessPoolExecutor):       {t_b:.2f}s")
print(f"  Latencia promedio: {sum(r['latencia'] for r in res_b)/N_USUARIOS:.2f}s/usuario")
print()
print(f"Speedup b/a: {t_a/t_b:.2f}x")
print()
S_total = T_IO + T_CPU
fraccion_serial = T_IO / S_total
p = os.cpu_count()
speedup_amdahl = 1 / (fraccion_serial + (1 - fraccion_serial) / p)
print(f"Ley de Amdahl: fracción serial s={fraccion_serial:.2f}, P={p} cores")
print(f"  Speedup teórico máximo: {speedup_amdahl:.2f}x")
print(f"  La fracción I/O ({T_IO}s de {S_total}s total) limita el speedup.")